In [7]:
# ==============================================================================
# [GLOBAL CELL 1] KONFIGURASI SENTRAL & SEEDING (POST-TRAINING QUANTIZATION)
# ==============================================================================
import os
import random
from enum import Enum
from typing import Tuple, List

import numpy as np
import tensorflow as tf

class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'

class PTQConfig:
    """Konfigurasi utama untuk Fase Post-Training Quantization (PTQ) INT8 (Ablation Study)."""
    
    PATH_TRAIN_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/train"
    PATH_VAL_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/val"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    
    # Mode Debug (Ganti ke 64 untuk production)
    SAMPLES_PER_CLASS: int = 1  
    EVAL_BATCH_SIZE: int = 1   
    
    # --------------------------------------------------------------------------
    # ABLATION STUDY: PATH INTEGRASI DARI 2 FASE BERBEDA
    # --------------------------------------------------------------------------
    PATH_BASELINE_MODELS: str = "/kaggle/input/training-outputs" # Sesuaikan path ini
    PATH_RECOVERED_MODELS: str = "/kaggle/input/datasets/marrioqqqq/recovery-outputs"
    
    BASE_OUTPUT_DIR: str = "/kaggle/working/ptq_outputs"

    @classmethod
    def get_output_dir(cls, arch: ModelArch) -> str:
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def get_input_model_path(cls, arch: ModelArch, model_type: str) -> str:
        """Mengambil model .keras berdasarkan tipe (baseline atau pruned)."""
        if model_type == 'baseline':
            base_path = os.path.join(cls.PATH_BASELINE_MODELS, arch.value)
            if not os.path.exists(base_path): base_path = cls.PATH_BASELINE_MODELS
            for file in os.listdir(base_path):
                if file.endswith("_phase2.keras") and arch.value in file:
                    return os.path.join(base_path, file)
        else:
            base_path = os.path.join(cls.PATH_RECOVERED_MODELS, arch.value)
            if not os.path.exists(base_path): base_path = cls.PATH_RECOVERED_MODELS
            for file in os.listdir(base_path):
                if file.endswith("_recovered.keras") and arch.value in file:
                    return os.path.join(base_path, file)
                    
        raise FileNotFoundError(f"[ERROR] Model {model_type} untuk {arch.value} tidak ditemukan.")

    @classmethod
    def get_tflite_path(cls, arch: ModelArch, model_type: str) -> str:
        """Memisahkan nama file output INT8 antara baseline dan pruned."""
        return os.path.join(cls.get_output_dir(arch), f"model_{arch.value}_{model_type}_full_int8.tflite")

    @classmethod
    def get_db_path(cls, arch: ModelArch) -> str:
        return os.path.join(cls.get_output_dir(arch), f"ptq_analytics_{arch.value}.db")

def set_deterministic_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

set_deterministic_seed(42)

[SETUP] Deterministic Seed dikunci pada: 42
[SETUP] TensorFlow Version: 2.20.0


In [8]:
# ==============================================================================
# [CELL 2] STRATIFIED CALIBRATION UTILITY & PTQ SQLITE LOGGER
# ==============================================================================
import sqlite3
from typing import Dict, Any, Callable, Generator
from tensorflow.keras import layers

def patch_keras_serialization() -> None:
    if hasattr(layers.Conv2D, '_is_patched_for_pruning'): return
    _orig_dense = layers.Dense.__init__; _orig_conv2d = layers.Conv2D.__init__
    _orig_dwconv2d = layers.DepthwiseConv2D.__init__; _orig_bn = layers.BatchNormalization.__init__

    def safe_dense(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_dense(self, *args, **kwargs)
    def safe_conv2d(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_conv2d(self, *args, **kwargs)
    def safe_dwconv2d(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_dwconv2d(self, *args, **kwargs)
    def safe_bn(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_bn(self, *args, **kwargs)

    layers.Dense.__init__ = safe_dense; layers.Conv2D.__init__ = safe_conv2d
    layers.DepthwiseConv2D.__init__ = safe_dwconv2d; layers.BatchNormalization.__init__ = safe_bn
    layers.Conv2D._is_patched_for_pruning = True

patch_keras_serialization()

class StratifiedCalibrationBuilder:
    @staticmethod
    def build_generator(data_dir: str, img_size: Tuple[int, int], samples_per_class: int, seed: int = 42) -> Callable[[], Generator]:
        random.seed(seed)
        classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        subset_paths = []
        
        print(f"\n[INFO] Membangun Stratified Calibration Dataset ({samples_per_class} img/class):")
        for cls_name in classes:
            cls_dir = os.path.join(data_dir, cls_name)
            img_paths = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            random.shuffle(img_paths)
            selected_paths = img_paths[:samples_per_class]
            subset_paths.extend(selected_paths)
            print(f"  -> Kelas '{cls_name}': {len(selected_paths)} sampel diambil.")
            
        random.shuffle(subset_paths)
        def representative_data_gen():
            for img_path in subset_paths:
                img = tf.io.read_file(img_path)
                img = tf.image.decode_jpeg(img, channels=3)
                img = tf.image.resize(img, img_size)
                img = tf.expand_dims(img, axis=0)
                img = tf.cast(img, tf.float32)
                yield [img]
        return representative_data_gen

class PTQSQLiteLogger:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._initialize_db()

    def _initialize_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Penambahan Primary Key gabungan (architecture, model_type)
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS ptq_macro_metrics (
                architecture TEXT, model_type TEXT,
                size_fp32_mb REAL, size_int8_mb REAL, memory_reduction_pct REAL,
                accuracy_fp32 REAL, accuracy_int8 REAL, quantization_error REAL,
                PRIMARY KEY (architecture, model_type)
            )
        ''')
        
        # Penambahan kolom model_type
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS ptq_micro_analytics (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                model_type TEXT, keras_layer_name TEXT, tflite_tensor_name TEXT, shape TEXT,
                scale REAL, zero_point INTEGER, fp32_weight_mean REAL, 
                int8_reconstructed_mean REAL, mean_shift_error REAL
            )
        ''')
        conn.commit(); conn.close()

    def log_macro(self, data: Dict[str, Any]) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO ptq_macro_metrics 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            data['Architecture'], data['Model_Type'], data['Size_FP32_MB'], 
            data['Size_INT8_MB'], data['Memory_Reduction_Pct'], 
            data['Accuracy_FP32'], data['Accuracy_INT8'], data['Quantization_Error']
        ))
        conn.commit(); conn.close()

    def log_micro_batch(self, analytics_list: List[Dict[str, Any]], model_type: str) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("DELETE FROM ptq_micro_analytics WHERE model_type = ?", (model_type,))
        
        records = [(
            model_type, a['Matched_Keras_Layer'], a['TFLite_Tensor_Name'], a['Shape'],
            a['Scale'], a['Zero_Point'], a['FP32_Weight_Mean'],
            a['Reconstructed_INT8_Mean'], a['Mean_Shift_Error']
        ) for a in analytics_list]
        
        cursor.executemany('''
            INSERT INTO ptq_micro_analytics 
            (model_type, keras_layer_name, tflite_tensor_name, shape, scale, zero_point, 
             fp32_weight_mean, int8_reconstructed_mean, mean_shift_error)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', records)
        conn.commit(); conn.close()

[BERHASIL] Stratified Calibration Utility & PTQ SQLite siap.


In [9]:
# ==============================================================================
# [CELL 3] PTQ ENGINE: CONVERSION, DEEP ANALYTICS, & EVALUATION
# ==============================================================================
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import models
from typing import Dict, Any, List

class PTQEngine:
    def __init__(self, config: PTQConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.db_path = self.cfg.get_db_path(self.arch)
        self.logger = PTQSQLiteLogger(self.db_path)

    def _convert_to_tflite(self) -> None:
        print(f"[INFO] Mengonfigurasi TFLite Converter untuk KOMPRESI EKSTREM (Full INT8)...")
        converter = tf.lite.TFLiteConverter.from_keras_model(self.fp32_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
        rep_gen = StratifiedCalibrationBuilder.build_generator(
            data_dir=self.cfg.PATH_TRAIN_FFB, img_size=self.cfg.IMG_SIZE, samples_per_class=self.cfg.SAMPLES_PER_CLASS
        )
        converter.representative_dataset = rep_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8  
        converter.inference_output_type = tf.uint8

        print("[INFO] Mengeksekusi kalibrasi dan konversi TFLite...")
        tflite_model_int8 = converter.convert()
        
        with open(self.tflite_path, 'wb') as f:
            f.write(tflite_model_int8)
        print(f"[BERHASIL] Model Full INT8 disimpan ke: {self.tflite_path}")

    def _extract_layerwise_analytics(self) -> None:
        print("\n[INFO] Mengekstrak Data Transparansi Kuantisasi per Layer...")
        interpreter = tf.lite.Interpreter(model_path=self.tflite_path)
        interpreter.allocate_tensors()
        tensor_details = interpreter.get_tensor_details()
        
        analytics_list: List[Dict[str, Any]] = []
        keras_layer_names = [layer.name for layer in self.fp32_model.layers if len(layer.get_weights()) > 0]
        
        for detail in tensor_details:
            if len(detail['shape']) >= 2 and detail['dtype'] == np.int8:
                quant_params = detail.get('quantization_parameters', {})
                scale = quant_params.get('scales', [0.0])[0] if len(quant_params.get('scales', [])) > 0 else 0.0
                zero_point = quant_params.get('zero_points', [0])[0] if len(quant_params.get('zero_points', [])) > 0 else 0
                
                matched_keras_name = "Unknown"
                fp32_mean = 0.0
                for k_name in keras_layer_names:
                    if k_name in detail['name']:
                        matched_keras_name = k_name
                        fp32_weights = self.fp32_model.get_layer(k_name).get_weights()[0]
                        fp32_mean = float(np.mean(fp32_weights))
                        break
                
                if matched_keras_name == "Unknown": continue
                
                try:
                    int8_weights = interpreter.get_tensor(detail['index'])
                    dequantized_weights = (int8_weights.astype(np.float32) - zero_point) * scale
                    int8_mean_reconstructed = float(np.mean(dequantized_weights))
                    mean_shift_error = abs(fp32_mean - int8_mean_reconstructed)
                except ValueError:
                    int8_mean_reconstructed, mean_shift_error = 0.0, 0.0
                
                analytics_list.append({
                    'Matched_Keras_Layer': matched_keras_name, 'TFLite_Tensor_Name': detail['name'],
                    'Shape': str(detail['shape']), 'Scale': round(scale, 6), 'Zero_Point': zero_point,
                    'FP32_Weight_Mean': round(fp32_mean, 6), 'Reconstructed_INT8_Mean': round(int8_mean_reconstructed, 6),
                    'Mean_Shift_Error': round(mean_shift_error, 8)
                })
                
        self.logger.log_micro_batch(analytics_list, self.current_model_type)
        print(f"[INFO] Terekstraksi {len(analytics_list)} layer bobot. Data tersimpan di SQLite.")

    def _evaluate_accuracies(self) -> None:
        print(f"\n[INFO] Menjalankan Evaluasi Akurasi Validasi (FP32 vs INT8)...")
        val_ds = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_VAL_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=self.cfg.EVAL_BATCH_SIZE, shuffle=False, label_mode='int', verbose=0
        ).prefetch(buffer_size=tf.data.AUTOTUNE)
        
        _, fp32_accuracy = self.fp32_model.evaluate(val_ds, verbose=0)
        
        interpreter = tf.lite.Interpreter(model_path=self.tflite_path)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()[0]
        output_details = interpreter.get_output_details()[0]
        
        correct, total = 0, 0
        for images, labels in val_ds:
            for i in range(len(images)):
                input_image = tf.expand_dims(images[i], axis=0)
                scale, zero_point = input_details['quantization']
                if scale > 0.0: input_image = input_image / scale + zero_point
                input_image = tf.cast(tf.math.round(input_image), input_details['dtype'])
                
                interpreter.set_tensor(input_details['index'], input_image)
                interpreter.invoke()
                output_data = interpreter.get_tensor(output_details['index'])
                if np.argmax(output_data[0]) == labels[i].numpy(): correct += 1
                total += 1
                
        int8_accuracy = correct / total
        quantization_error = fp32_accuracy - int8_accuracy
        size_fp32_mb = os.path.getsize(self.fp32_path) / (1024 * 1024)
        size_int8_mb = os.path.getsize(self.tflite_path) / (1024 * 1024)
        memory_reduction = (1 - (size_int8_mb / size_fp32_mb)) * 100
        
        print("\n" + "="*50)
        print(f"📊 PROFIL KOMPRESI AKHIR ({self.arch.value} - {self.current_model_type.upper()})")
        print("="*50)
        print(f"1. Memory Footprint : {size_fp32_mb:.2f} MB -> {size_int8_mb:.2f} MB (-{memory_reduction:.1f}%)")
        print(f"2. FP32 Accuracy    : {fp32_accuracy*100:.2f}%")
        print(f"3. INT8 Accuracy    : {int8_accuracy*100:.2f}%")
        print(f"4. Quantization Drop: {quantization_error*100:.2f}%")
        print("="*50)
        
        self.logger.log_macro({
            'Architecture': self.arch.value, 'Model_Type': self.current_model_type,
            'Size_FP32_MB': round(size_fp32_mb, 2), 'Size_INT8_MB': round(size_int8_mb, 2),
            'Memory_Reduction_Pct': round(memory_reduction, 2), 'Accuracy_FP32': round(fp32_accuracy, 4),
            'Accuracy_INT8': round(int8_accuracy, 4), 'Quantization_Error': round(quantization_error, 4)
        })

    def execute_ptq(self) -> None:
        """Mengeksekusi kuantisasi pada dua tipe model secara berurutan untuk Ablation Study."""
        for model_type in ['baseline', 'pruned']:
            print(f"\n" + "❖"*60)
            print(f"🔄 MEMPROSES MODEL: {model_type.upper()} ({self.arch.value})")
            print("❖"*60)
            
            self.current_model_type = model_type
            self.fp32_path = self.cfg.get_input_model_path(self.arch, model_type)
            self.tflite_path = self.cfg.get_tflite_path(self.arch, model_type)
            
            self.fp32_model = models.load_model(self.fp32_path)
            self.fp32_model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
            
            self._convert_to_tflite()
            self._extract_layerwise_analytics()
            self._evaluate_accuracies()

[BERHASIL] Cell 3 (PTQ Engine) dimuat.


In [10]:
# ==============================================================================
# [CELL 4] PTQ EXECUTOR: BATCH LOOP MODE
# ==============================================================================
import gc
import tensorflow as tf

config = PTQConfig()

# Seleksi Arsitektur
MODELS_TO_QUANTIZE = [
    ModelArch.MOBILENET_V3,
    # ModelArch.EFFICIENTNET_B0,  # Buka pagar (uncomment) jika ingin diuji sekaligus
]

print("\n" + "★"*75)
print(f"🔧 MEMULAI POST-TRAINING QUANTIZATION (INT8) UNTUK {len(MODELS_TO_QUANTIZE)} ARSITEKTUR")
print("★"*75)

for idx, arch in enumerate(MODELS_TO_QUANTIZE, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_QUANTIZE)}] EKSEKUSI PTQ: {arch.value}")
    print(f"{'='*75}")
    
    try:
        # Reset State & Seed untuk menghindari konflik nama tensor atau kebocoran memori (Standar 3 & 6)
        tf.keras.backend.clear_session()
        gc.collect()
        set_deterministic_seed(42)
        
        # Inisialisasi & Eksekusi Mesin PTQ
        engine = PTQEngine(config=config, architecture=arch)
        engine.execute_ptq()
        
        print(f"\n[CLEANUP] Membersihkan ruang memori pasca-kuantisasi untuk {arch.value}...")
        del engine
        gc.collect()
        
    except FileNotFoundError as e:
        print(f"\n[SKIP] Melompati {arch.value} karena dependensi hilang: {e}")
    except RuntimeError as e:
        print(f"\n[CRITICAL ERROR] Kegagalan komputasi TFLite pada {arch.value}: {e}")
    except Exception as e:
        print(f"\n[UNKNOWN ERROR] Eksekusi PTQ terhenti secara tak terduga pada {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES POST-TRAINING QUANTIZATION SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔧 MEMULAI POST-TRAINING QUANTIZATION (INT8) UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

[1/1] EKSEKUSI PTQ: MobileNetV3Small
[SETUP] Deterministic Seed dikunci pada: 42
[INFO] Memuat Model FP32 Pareto-Optimal: /kaggle/input/datasets/marrioqqqq/recovery-outputs/MobileNetV3Small/model_MobileNetV3Small_pruned_40pct_recovered.keras


2026-07-22 02:27:07.405492: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[INFO] Mengonfigurasi TFLite Converter untuk KOMPRESI EKSTREM (Full INT8)...

[INFO] Membangun Stratified Calibration Dataset (1 img/class):
  -> Kelas 'lewat_matang': 1 sampel diambil.
  -> Kelas 'matang': 1 sampel diambil.
  -> Kelas 'mentah': 1 sampel diambil.
  -> Total Representative Dataset: 3 citra.
[INFO] Mengeksekusi kalibrasi dan konversi TFLite...
INFO:tensorflow:Assets written to: /tmp/tmpazoudlf3/assets


INFO:tensorflow:Assets written to: /tmp/tmpazoudlf3/assets


Saved artifact at '/tmp/tmpazoudlf3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_ffb')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133636593008080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591944336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591945296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636593007696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636593007888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591945680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591945872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591944720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591945488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591946448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133636591946832: 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1784687236.141978      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784687236.142039      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784687236.300485      58 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WAR

[BERHASIL] Model Full INT8 disimpan ke: /kaggle/working/ptq_outputs/MobileNetV3Small/model_MobileNetV3Small_full_int8.tflite

[INFO] Mengekstrak Data Transparansi Kuantisasi per Layer...
[INFO] Terekstraksi 77 layer bobot. Data tersimpan di SQLite.

[INFO] Menjalankan Evaluasi Akurasi Validasi (FP32 vs INT8)...

📊 PROFIL KOMPRESI AKHIR (MobileNetV3Small)
1. Memory Footprint : 4.49 MB -> 0.69 MB (-84.7%)
2. FP32 Accuracy    : 33.33%
3. INT8 Accuracy    : 33.33%
4. Quantization Drop: 0.00%

[CLEANUP] Membersihkan ruang memori pasca-kuantisasi untuk MobileNetV3Small...

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PROSES POST-TRAINING QUANTIZATION SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [11]:
# ==============================================================================
# [CELL 5] PTQ VISUALIZER: ABLATION STUDY COMPARISONS
# ==============================================================================
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

class PTQVisualizer:
    def __init__(self, config: PTQConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.db_path = self.cfg.get_db_path(self.arch)
        self.vis_dir = os.path.join(self.cfg.get_output_dir(self.arch), "visualisation")
        os.makedirs(self.vis_dir, exist_ok=True)
        
        self.p_macro = os.path.join(self.vis_dir, f"ptq_01_{self.arch.value}_ablation_macro.png")
        self.p_micro = os.path.join(self.vis_dir, f"ptq_02_{self.arch.value}_pruned_micro.png")
        
        self._validate_database()
        self.df_macro = self._load_data("ptq_macro_metrics")
        self.df_micro = self._load_data("ptq_micro_analytics")

    def _validate_database(self) -> None:
        if not os.path.exists(self.db_path):
            raise FileNotFoundError(f"[ERROR] Database {self.db_path} tidak ditemukan.")
            
    def _load_data(self, table_name: str) -> pd.DataFrame:
        conn = sqlite3.connect(self.db_path)
        try:
            if table_name == "ptq_macro_metrics":
                df = pd.read_sql_query(f"SELECT * FROM {table_name} WHERE architecture = '{self.arch.value}'", conn)
            else:
                df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        except sqlite3.OperationalError:
            df = pd.DataFrame()
        conn.close()
        return df

    def plot_macro_impact(self) -> None:
        if self.df_macro.empty: return
        
        plt.style.use('default')
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # Mengurai 2 baris (baseline dan pruned) menjadi 4 kategori label
        row_base = self.df_macro[self.df_macro['model_type'] == 'baseline'].iloc[0]
        row_pruned = self.df_macro[self.df_macro['model_type'] == 'pruned'].iloc[0]

        labels = ['Base (FP32)', 'Base (INT8)', 'Pruned (FP32)', 'Pruned (INT8)']
        sizes_mb = [row_base['size_fp32_mb'], row_base['size_int8_mb'], row_pruned['size_fp32_mb'], row_pruned['size_int8_mb']]
        accuracies = [row_base['accuracy_fp32']*100, row_base['accuracy_int8']*100, row_pruned['accuracy_fp32']*100, row_pruned['accuracy_int8']*100]

        # Bar Chart Memory
        color_size = '#34495e'
        x_pos = np.arange(len(labels))
        bars = ax1.bar(x_pos, sizes_mb, width=0.4, color=color_size, alpha=0.8)
        
        ax1.set_ylabel('Model Size (MB)', color=color_size, fontweight='bold')
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels(labels, fontweight='bold')
        ax1.set_ylim(0, max(sizes_mb) * 1.3)

        for bar in bars:
            yval = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f} MB', ha='center', va='bottom', fontweight='bold')

        # Line Plot Accuracy
        ax2 = ax1.twinx()
        color_acc = '#e74c3c'
        ax2.plot(x_pos, accuracies, color=color_acc, marker='o', linestyle='-', linewidth=3, markersize=10)
        ax2.set_ylabel('Accuracy (%)', color=color_acc, fontweight='bold')
        
        padding = max(5.0, (max(accuracies) - min(accuracies)) * 0.5) 
        ax2.set_ylim(max(0, min(accuracies) - padding), min(100.5, max(accuracies) + padding))

        for i, acc in enumerate(accuracies):
            ax2.annotate(f'{acc:.2f}%', (x_pos[i], acc), textcoords="offset points", xytext=(0, -20), ha='center', fontweight='bold', color='darkred')

        plt.title(f"Ablation Study Kuantisasi: {self.arch.value}", pad=15, fontweight='bold')
        fig.tight_layout()
        plt.savefig(self.p_macro, dpi=300, bbox_inches='tight')
        plt.close()

    def plot_micro_analytics(self) -> None:
        """Memplot grafik distorsi Mean Shift Error secara spesifik untuk model yang Pruned."""
        df_pruned = self.df_micro[self.df_micro['model_type'] == 'pruned']
        if df_pruned.empty: return

        df_sorted = df_pruned.sort_values(by='mean_shift_error', ascending=False).head(15)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        sns.barplot(data=df_sorted, x='mean_shift_error', y='keras_layer_name', ax=ax1, palette='rocket', hue='keras_layer_name', legend=False)
        ax1.set_title(f'Top 15 Layer Distorsi (Pruned Model)', fontweight='bold')
        ax1.set_xlabel('Mean Shift Error'); ax1.set_ylabel('Nama Layer Konvolusi')

        sns.scatterplot(data=df_pruned, x='fp32_weight_mean', y='int8_reconstructed_mean', ax=ax2, color='#2980b9', alpha=0.7)
        min_val = min(df_pruned['fp32_weight_mean'].min(), df_pruned['int8_reconstructed_mean'].min())
        max_val = max(df_pruned['fp32_weight_mean'].max(), df_pruned['int8_reconstructed_mean'].max())
        ax2.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Ideal Fit (y=x)')
        
        ax2.set_title('Korelasi Presisi Bobot (Pruned Model)', fontweight='bold')
        ax2.set_xlabel('Rata-rata Bobot FP32'); ax2.set_ylabel('Rata-rata Bobot INT8 (Rekonstruksi)')
        ax2.legend()

        fig.tight_layout()
        plt.savefig(self.p_micro, dpi=300, bbox_inches='tight')
        plt.close()

    def run_all_visualisations(self) -> None:
        self.plot_macro_impact()
        self.plot_micro_analytics()
        print(f"[BERHASIL] Gambar Resolusi Tinggi Tersimpan di: {self.vis_dir}")
        display(FileLink(self.p_macro))
        display(FileLink(self.p_micro))

[BERHASIL] Cell 5 (Class PTQVisualizer) dimuat.


In [12]:
# ==============================================================================
# [CELL 6] PTQ VISUALIZATION EXECUTOR (BATCH RUNNER)
# ==============================================================================

# Mengambil variabel MODELS_TO_QUANTIZE dari Cell 4 jika ada di memori.
# Jika tidak (misal kernel di-restart), fallback ke list default.
try:
    models_to_visualize = MODELS_TO_QUANTIZE
except NameError:
    models_to_visualize = [ModelArch.MOBILENET_V3] 

print("\n" + "★"*75)
print(f"🎨 MEMULAI RENDER VISUALISASI PTQ UNTUK {len(models_to_visualize)} ARSITEKTUR")
print("★"*75)

config_vis = PTQConfig()

for arch in models_to_visualize:
    try:
        # Inisialisasi visualizer untuk arsitektur spesifik
        visualizer = PTQVisualizer(config=config_vis, architecture=arch)
        
        # Render dan simpan grafik Makro & Mikro
        visualizer.run_all_visualisations()
        
    except FileNotFoundError as e:
        print(e)
    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Gagal merender grafik PTQ untuk {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES RENDER GRAFIK KUANTISASI SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🎨 MEMULAI RENDER VISUALISASI PTQ UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📊 INISIALISASI VISUALIZER PTQ: MobileNetV3Small
[BERHASIL] Gambar Resolusi Tinggi Tersimpan di: /kaggle/working/ptq_outputs/MobileNetV3Small/visualisation


/kaggle/working/ptq_outputs/MobileNetV3Small/visualisation/ptq_01_MobileNetV3Small_macro_tradeoff.png

/kaggle/working/ptq_outputs/MobileNetV3Small/visualisation/ptq_02_MobileNetV3Small_micro_distortion.png


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PROSES RENDER GRAFIK KUANTISASI SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
